# 01 环境与数据

对应教程章节：[02-环境搭建](../02-环境搭建.md)、[03-数据准备](../03-数据准备.md)

**本 notebook 的终点**：环境三条命令通过、看懂标签、把数据切成 train/val/test。

> **运行前提**：Jupyter 右上角的内核要选 **ai-env**（里面才有 torch 和 ultralytics）。
> 如果选不到，先在终端执行：
> ```bash
> conda activate ai-env
> pip install ipykernel matplotlib
> python -m ipykernel install --user --name ai-env --display-name "Python (ai-env)"
> ```
> 然后刷新页面、重新选内核。

**怎么用**：从上往下逐格运行（`Shift+Enter`）。每格都是幂等的，重复运行不会出问题。

## 1. 环境自检

In [1]:
# ===== 环境自检与路径准备：每个 notebook 都先运行这一格 =====
import os, sys, pathlib, subprocess

# 0) 记下 notebook 启动时所在目录（后面用它找配套脚本），再切到作业根目录
START_DIR = pathlib.Path.cwd()

# 1) 缓存目录指到可写位置，避免 matplotlib / ultralytics 的权限报错
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplcache")
os.environ.setdefault("YOLO_CONFIG_DIR", "/tmp/ultralytics")

# 2) 定位作业根目录：向上查找含 data/ 或 datasets/ 或 codebase/ 的目录
def find_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for cand in (p, *p.parents):
        if any((cand / d).is_dir() for d in ("data", "datasets", "codebase")):
            return cand
    return p

ROOT = find_root()
os.chdir(ROOT)
print("作业根目录:", ROOT)

# 3) 与配套脚本一致：优先使用作业自带的 ultralytics 源码
CB = ROOT / "codebase" / "ultralytics-YOLO26"
if CB.is_dir() and str(CB) not in sys.path:
    sys.path.insert(0, str(CB))

# 4) 依赖与 GPU 自检
missing = []
for mod, name in (("torch", "PyTorch"), ("ultralytics", "ultralytics"),
                  ("PIL", "Pillow"), ("matplotlib", "matplotlib")):
    try:
        __import__(mod)
    except ImportError:
        missing.append(name)
if missing:
    print("❌ 缺少依赖:", missing)
    print("   请在 ai-env 里执行：conda activate ai-env && pip install ipykernel matplotlib")
else:
    import torch, ultralytics
    print("✅ 依赖齐全")
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "（无 GPU，训练会非常慢）"
    print("CUDA 可用:", torch.cuda.is_available(), "|", gpu)
    print("ultralytics:", ultralytics.__version__, "->", ultralytics.__file__)


作业根目录: /home/tomliu/Programs/assignment4
✅ 依赖齐全
CUDA 可用: True | NVIDIA GeForce RTX 5060 Laptop GPU
ultralytics: 8.4.7 -> /home/tomliu/Programs/assignment4/codebase/ultralytics-YOLO26/ultralytics/__init__.py


In [ ]:
# ===== 工具函数：找脚本 / 运行脚本 / 显示图片 =====
# 画图后端：在 Jupyter / IPython 里必须用 inline，否则 plt.show() 不会把图显示出来
# （纯脚本环境没有 IPython，就退回 Agg —— 不弹窗、只保存文件）
try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def script(name):
    """定位配套脚本，兼容三种常见目录布局：

    1. <作业根>/code/train.py      —— 教程默认布局（把 code/ 拷进作业根）
    2. <作业根>/train.py           —— 把脚本直接放在作业根
    3. <notebook 同级的 code/>     —— 直接在教程文件夹里运行 notebook
    """
    cands = [ROOT / "code" / name, ROOT / name,
             START_DIR.parent / "code" / name, START_DIR / "code" / name]
    for c in cands:
        if c.exists():
            return str(c)
    raise FileNotFoundError(
        f"找不到 {name}。请把教程的 code/ 目录复制到作业根目录（与 data/ 同级），"
        f"或把脚本直接放在作业根目录。已查找：{[str(c) for c in cands]}")

def run(*args, check=True):
    """用当前内核的 Python 运行配套脚本，输出直接显示在下面。"""
    cmd = [sys.executable, *[str(a) for a in args]]
    print("$", " ".join(cmd), "\n")
    return subprocess.run(cmd, check=check)

def show(path, width=800, title=None):
    """在 notebook 里显示一张本地图片。

    图片文件直接用 IPython.display 显示（不经过 matplotlib，最稳）；
    不在 Jupyter 里时退回 PIL + matplotlib，至少不会报错。
    """
    path = pathlib.Path(path)
    if not path.exists():
        print("⚠️ 文件不存在:", path, "（先运行生成它的那一格）")
        return
    if title:
        print(title)
    try:
        from IPython.display import Image as _IPyImage, display
        display(_IPyImage(filename=str(path), width=width))
        return
    except ImportError:
        pass
    img = Image.open(path)
    w, h = img.size
    fig, ax = plt.subplots(figsize=(8, 8 * h / w))
    ax.imshow(img); ax.axis("off")
    if title:
        ax.set_title(title)
    plt.show()


如果上面提示缺依赖或 CUDA 不可用，先解决再往下走——后面的报错会很难定位。
下面这一格看显卡：**显存决定后面 batch 能开多大**。

In [5]:
out = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total,driver_version,power.limit,power.max_limit --format=csv",
    shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)

name, memory.total [MiB], driver_version, power.limit [W], power.max_limit [W]
NVIDIA GeForce RTX 5060 Laptop GPU, 8151 MiB, 595.91.07, [N/A], 115.00 W



> **参考值**：RTX 5060 Laptop，8151 MiB，驱动 595.91.07，功耗上限 115W。
> 如果 `power.limit` 明显低于 `power.max_limit`（例如 50W vs 115W），说明显卡被锁在低功耗档，
> 训练会慢一倍——解决办法见 [07-排错手册](../07-排错手册.md) 的"功耗被锁"案例。

## 2. 最小推理验证链路

正式训练前，先用一张示例图确认"Python + PyTorch + CUDA + ultralytics + 预训练权重"这条链是通的。

In [7]:
import torch
from ultralytics import YOLO

CB = ROOT / "codebase" / "ultralytics-YOLO26"
asset = CB / "ultralytics" / "assets" / "bus.jpg"
m = YOLO(str(CB / "yolo26n.pt"))
r = m.predict(str(asset), imgsz=640, conf=0.4,
              device=0 if torch.cuda.is_available() else "cpu", verbose=False)[0]
print("检测到", len(r.boxes), "个目标：",
      [(m.names[int(c)], round(float(cf), 2)) for c, cf in zip(r.boxes.cls, r.boxes.conf)])

# 把带框的结果画出来（优先用 IPython 显示 PIL 图，不依赖 matplotlib 后端）
from PIL import Image as _PILImage
_pil = _PILImage.fromarray(r.plot()[:, :, ::-1])
try:
    from IPython.display import display
    display(_pil)
except ImportError:                     # 非 notebook 环境
    plt.figure(figsize=(9, 6)); plt.imshow(_pil); plt.axis("off"); plt.show()

检测到 5 个目标： [('bus', 0.92), ('person', 0.91), ('person', 0.91), ('person', 0.87), ('person', 0.54)]


/tmp/ipykernel_8115/1515644515.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off"); plt.show()


✅ 能看到公交车和行人被框出来，说明链路是通的。

⚠️ 注意：这个模型只认识 COCO 的 80 类，**不认识机器人**——认不出机器人是正常的，下一步就用我们自己的数据微调它。

## 3. 数据体检

拿到数据先做四件事：**图片与标签是否成对、坐标是否越界、类别编号有哪些、每张图多少目标**。
数据决定模型上限，这步比调参重要得多。

In [8]:
img_dir, lab_dir = ROOT / "data/labeled/images", ROOT / "data/labeled/labels"

imgs = {p.stem for p in img_dir.glob("*.jpg")}
labs = {p.stem for p in lab_dir.glob("*.txt")}
print("已标注：图片", len(imgs), "| 标签", len(labs))
print("缺标签的图:", sorted(imgs - labs)[:5] or "无")
print("缺图的标签:", sorted(labs - imgs)[:5] or "无")

classes, boxes, per_image = set(), 0, []
for f in sorted(lab_dir.glob("*.txt")):
    lines = [l for l in f.read_text().splitlines() if l.strip()]
    per_image.append(len(lines))
    for l in lines:
        v = l.split()
        classes.add(v[0])
        assert all(0 <= float(x) <= 1 for x in v[1:5]), f"坐标越界: {f.name}"
    boxes += len(lines)

print("类别编号:", sorted(classes), "（本次只有 0 一类，即 robot）")
print(f"总框数 {boxes} | 平均 {boxes/len(per_image):.2f} 框/图 | 单图最多 {max(per_image)} 个")
print("空标签文件数（无目标的图）:", sum(1 for n in per_image if n == 0))

已标注：图片 1511 | 标签 1511
缺标签的图: 无
缺图的标签: 无
类别编号: ['0'] （本次只有 0 一类，即 robot）
总框数 6855 | 平均 4.54 框/图 | 单图最多 16 个
空标签文件数（无目标的图）: 0


**参考值**：1511 张图、6855 个框、平均 4.54 框/图、单图最多 16 个、没有空标签。
这些数字后面有用——模型输出"平均 4.5 框/图"才说明它学到了正确的目标密度。

### 3.1 把标签画出来亲眼核对（notebook 相比纯命令行最有用的一步）

随机抽 6 张，把标签框画在图上。**标注错了，在这里一眼就能看出来。**

In [ ]:
import random
random.seed(0)
picked = random.sample(sorted(labs), 6)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, stem in zip(axes.ravel(), picked):
    img = Image.open(img_dir / f"{stem}.jpg").convert("RGB")
    W, H = img.size
    d = ImageDraw.Draw(img)
    for line in (lab_dir / f"{stem}.txt").read_text().splitlines():
        if not line.strip():
            continue
        _, cx, cy, w, h = [float(v) for v in line.split()]
        d.rectangle([(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H],
                    outline=(255, 0, 0), width=8)
    ax.imshow(img); ax.set_title(f"{stem}.jpg", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

看懂这格需要知道标签格式：每行是 `类别 中心点x 中心点y 宽 高`，后四个数**已归一化**（除以图片宽高），
所以画框要换算成 `(cx-w/2)*W`。详细推导见 [03-数据准备](../03-数据准备.md) 3.2 节。

**观察重点**：红框是否紧贴机器人？有没有框到背景？有没有明显漏标？发现问题先修数据再训练。

## 4. 划分 train / val / test

分工：train 训练、val 挑权重和调参、**test 只在最后评估一次**（用 test 反复调参，成绩就不可信了）。

本次按"每张图的框数"分层划分：HUD 截图目标多、实拍图目标少，纯随机划分会让三个集合难度不一致，指标就会随划分抖动。

In [ ]:
run(script("prepare_data.py"), "--dry-run")   # 只看统计，不写文件

In [ ]:
run(script("prepare_data.py"))              # 真正划分，生成 datasets/rm/ 与 data.yaml

**检查点**：三个集合平均框数要接近（本次 4.52 / 4.59 / 4.56，差异 <5%）。差得多说明分层没生效。

## 5. 验证 data.yaml

In [ ]:
run("-c", "from ultralytics.data.utils import check_det_dataset; "
          "print(check_det_dataset('datasets/rm/data.yaml'))")

预期看到 `nc: 1`、`names: {0: 'robot'}`，以及 train/val/test 三个能解析的绝对路径。

### 关于未标注数据

数据里还有一批"只有图片、没有标签"的数据。**不要直接丢进训练集当背景**——这些图里通常有目标，
配空标签等于告诉模型"这里什么都没有"，会压低召回。正确用法（预标注 + 人工核对）见 notebook 03 与 [08-进阶与交付](../08-进阶与交付.md)。

## 检查点

- [ ] 环境自检没有缺依赖，CUDA 可用
- [ ] 最小推理跑通，图上能看到框
- [ ] 数据体检：图片与标签成对、坐标都在 0~1、类别只有 0
- [ ] 抽样的 6 张图里红框都框在目标上
- [ ] 划分完成，三个集合平均框数接近
- [ ] `check_det_dataset` 输出 `nc=1`

全部打勾后，继续 **02-训练与评估**。